### Performance comparada: antes de extrair, o modelo reproduz o sistema?

Junta, num só lugar, as métricas de teste (12 passos encadeados, escala original) de todos os experimentos já rodados: as 4 arquiteturas baseline + ARMAX (`mimo_sys_collection_70_15_15`), o ensemble e a cascata física (`mimo_sys_mass_balance_cascade`), e o MPNN treinado para as rotas A/B de extração (`mimo_sys_route_a_extraction`). O objetivo é responder antes de qualquer explicabilidade: **esse modelo de fato reproduz o sistema, ou só parece bom porque a tarefa que ele resolve é mais fácil?**

In [1]:
from pathlib import Path

import mlflow
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

MLFLOW_DB = Path("../training/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

OUTPUT_COLS = ["vazao_distribuicao", "nivel_res", "pressao"]

/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/16 19:53:27 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


### Carrega os 3 experimentos

Nenhum modelo é retreinado aqui — só leitura das métricas já registradas no MLflow.

In [2]:
baseline = mlflow.search_runs(
    experiment_names=["mimo_sys_collection_70_15_15"]
).rename(columns={"tags.mlflow.runName": "run"}).set_index("run")

cascade = mlflow.search_runs(
    experiment_names=["mimo_sys_mass_balance_cascade"]
).rename(columns={"tags.mlflow.runName": "run"}).set_index("run")

route_a = mlflow.search_runs(
    experiment_names=["mimo_sys_route_a_extraction"]
).rename(columns={"tags.mlflow.runName": "run"}).set_index("run")

print(f"baseline: {len(baseline)} runs | cascade: {len(cascade)} runs | route_a: {len(route_a)} runs")

baseline: 8 runs | cascade: 7 runs | route_a: 2 runs


### Tabela única: R² por saída, 12 passos genuínos

`baseline` e `cascade` treinam com `output_window=12` — a previsão do passo 12 nunca vê dado real entre o passo 1 e o 12, é regime autônomo de verdade. `route_a` usa `output_window=1`: a métrica `chained_r2` dele vem de refazer a previsão de 1 passo a cada novo ponto, sempre reancorada no dado real mais recente — ou seja, **não é a mesma tarefa**. As duas colunas ficam lado a lado de propósito, para isso pular aos olhos.

In [3]:
def r2_cols(df: pd.DataFrame, prefix: str, suffix: str) -> pd.DataFrame:
    cols = {out: f"metrics.{prefix}{out}{suffix}" for out in OUTPUT_COLS}
    cols = {k: v for k, v in cols.items() if v in df.columns}
    out = df[list(cols.values())].rename(columns={v: k for k, v in cols.items()})
    return out


rows = {
    "ARMAX 12 passos": r2_cols(baseline, "", "__r2").loc["ARMAX 12 passos"],
    "Seq2SeqAttention (baseline)": r2_cols(baseline, "", "__r2").loc["Seq2SeqAttention"],
    "StemGNN (baseline)": r2_cols(baseline, "", "__r2").loc["StemGNN"],
    "MPNN (baseline)": r2_cols(baseline, "", "__r2").loc["MPNN"],
    "SymbolicGraphNetwork (baseline)": r2_cols(baseline, "", "__r2").loc["SymbolicGraphNetwork"],
    "ensemble_ml (5 modelos)": r2_cols(cascade, "r2_", "").loc["ensemble_ml"],
    "ensemble_cascade (física)": r2_cols(cascade, "r2_", "").loc["ensemble_cascade"],
    "MPNN rota A (1 passo, reancorado)": r2_cols(route_a, "chained_r2_", "").loc["mpnn_route_a_final"],
}

comparison = pd.DataFrame(rows).T
comparison["r2_medio"] = comparison.mean(axis=1)
comparison.round(3)

,vazao_distribuicao,nivel_res,pressao,r2_medio
ARMAX 12 passos,0.569,0.285,0.144,0.333
Seq2SeqAttention (baseline),0.877,0.435,0.653,0.655
StemGNN (baseline),0.889,0.386,0.594,0.623
MPNN (baseline),0.909,0.248,0.667,0.608
SymbolicGraphNetwork (baseline),0.915,0.260,0.693,0.623
ensemble_ml (5 modelos),0.929,0.508,0.740,0.726
ensemble_cascade (física),0.929,0.758,0.740,0.809
"MPNN rota A (1 passo, reancorado)",0.929,0.915,0.674,0.839


### O que a tabela diz

- A meta do orientador é R² médio ≥ 0,85 com cada saída ≥ 0,8 em regime autônomo de 12 passos. **Nenhum modelo chega lá** — a cascata física é o melhor resultado real (R² médio 0,809), com `pressao` ainda abaixo de 0,8.
- O MPNN da rota A aparece com R² altíssimo em `nivel_res` (0,915) só porque a tarefa dele é outra: 1 passo à frente, sempre reancorado no dado real. Comparar esse número com o R² de 12 passos dos outros modelos é comparar maçã com laranja — e a célula seguinte mostra exatamente o tamanho do erro que isso escondia.
- **Esse é o ponto do pedido:** o MPNN usado nas rotas A/B de extração (`docs/changes/2026-10-16-rota-a-b-extracao.md`) nunca foi testado no regime de 12 passos que é o alvo real do projeto. Ele resolve bem um problema mais fácil, e é desse modelo fácil que a equação/subgrafo foram extraídos.

### MPNN da rota A, mas em 12 passos genuínos

Pra não ficar só na suspeita: roda o mesmo MPNN (pesos de `models/mpnn_route_a_final.pth`) em regime autônomo de 12 passos de verdade — a cada passo, as colunas exógenas (`freq_b*`, `vazao_entrada`) vêm do dado real, mas as 3 saídas (`vazao_distribuicao`, `nivel_res`, `pressao`) são realimentadas a partir da própria previsão do passo anterior, nunca do dado real — e compara com a métrica de 1 passo reancorado que ele reporta no MLflow.

In [4]:
import joblib
import numpy as np
import torch

from mimo_sys.architectures import MPNNForecaster
from mimo_sys.training.data import N_NODES, TARGET_IDX, load_splits
from mimo_sys.training.evaluation import metrics_table

preprocessor = joblib.load("../../models/mpnn_route_a_final_preprocessor.joblib")

model = MPNNForecaster(
    n_nodes=N_NODES,
    input_window=3,
    output_window=1,
    target_idx=TARGET_IDX,
    hidden_size=64,
    message_dim=2,
    n_layers=1,
)
model.load_state_dict(torch.load("../../models/mpnn_route_a_final.pth", map_location="cpu"))
model.eval()

test_raw = load_splits()["test"]
series = preprocessor.transform_series(test_raw)

INPUT_WINDOW, HORIZON = 3, 12
anchors = np.arange(INPUT_WINDOW, len(series) - HORIZON + 1, HORIZON)
outs = []
with torch.no_grad():
    for t0 in anchors:
        window = series[t0 - INPUT_WINDOW : t0].copy()
        steps = []
        for h in range(HORIZON):
            x = torch.as_tensor(window, dtype=torch.float32).unsqueeze(0)
            step_pred = model(x).squeeze(0).squeeze(0).numpy()  # (n_targets,)
            steps.append(step_pred)
            next_row = series[t0 + h].copy()  # true exogenous features
            next_row[TARGET_IDX] = step_pred  # fed back, not measured
            window = np.vstack([window[1:], next_row])
        outs.append(np.stack(steps))

pred_12 = preprocessor.inverse_transform(
    np.stack(outs), anchors, feats=TARGET_IDX, reference=test_raw
).reshape(-1, len(TARGET_IDX))
true_12 = test_raw[anchors[0] : anchors[0] + len(pred_12)][:, TARGET_IDX]
metrics_table(true_12, pred_12)

,mse,mae,r2,smape_%
vazao_distribuicao,269.979167,11.836122,0.566653,18.037366
nivel_res,0.119823,0.258209,-1.340419,11.036481
pressao,7.808572,1.941366,0.460972,9.151912


O resultado confirma a suspeita, e é mais grave do que "um pouco pior": em regime autônomo de 12 passos, `nivel_res` fica com **R² -1,34** (pior que prever a média), contra 0,915 no regime de 1 passo reancorado. `vazao_distribuicao` cai para 0,567 e `pressao` para 0,461. O MPNN da rota A **não reproduz o sistema** no regime que importa — ele só parece bom porque nunca foi obrigado a sustentar sua própria previsão por mais de 1 passo.

### Conclusão

Extrair equação ou subgrafo de um modelo só vale a pena se esse modelo reproduz o sistema no regime que importa (12 passos autônomos, meta R² ≥ 0,85). Hoje:

1. O melhor modelo em 12 passos é a **cascata física** (R² médio 0,809), não o MPNN da rota A.
2. O MPNN da rota A resolve bem uma tarefa mais fácil (1 passo reancorado) e, em regime autônomo de 12 passos, **não reproduz o sistema** — `nivel_res` fica com R² negativo (-1,34).
3. As equações e a topologia extraídas na nota `docs/changes/2026-10-16-rota-a-b-extracao.md` descrevem, portanto, **a dinâmica de 1 passo**, não a dinâmica de 12 passos que o projeto precisa explicar. A rota B (`edge_strength`) ainda é um resultado válido nesse recorte mais estreito — é topologia, não depende de horizonte —, mas a rota A (equação) herda a limitação do modelo de onde foi extraída.
4. Antes de insistir em extrair equação desse MPNN pequeno, falta treinar um MPNN **no mesmo regime de 12 passos** (`output_window=12`, mantendo `input_window` pequeno e `message_l1_weight` para a rota A continuar viável) e só então repetir a destilação simbólica.